## Implement Context Compaction

### Installing Utilities and Libraries

In [ ]:
%pip install \
    databricks-sdk==0.49.0 \
    anthropic==0.120.2 \
    "mlflow>=3.1"

### Restart your Python Environment

In [ ]:
dbutils.library.restartPython()

### Set up your Environment

In [ ]:
from databricks.sdk import WorkspaceClient

# Get Databricks runtime authentication
w = WorkspaceClient()

headers = w.config.authenticate()
token = headers["Authorization"].replace("Bearer ", "")
workspace_host = w.config.host.rstrip("/")

### Create the Anthropic Client

In [ ]:
import anthropic

# Anthropic client through Databricks
client = anthropic.Anthropic(
    api_key="unused",
    base_url=f"{workspace_host}/serving-endpoints/anthropic",
    default_headers={
        "Authorization": f"Bearer {token}"
    }
)

MODEL = "databricks-claude-sonnet-5"

### Define the Initial Context

In [ ]:
initial_prompt = """
You are helping me design an e-commerce platform called Project Falcon.

Remember these requirements throughout our conversation:

- Backend: Python
- Database: PostgreSQL
- Cache: Redis
- Cloud provider: Azure
- Authentication: Microsoft Entra ID
- Maximum infrastructure budget: $8,000/month
- Must support 100,000 concurrent users
- Customer payment information must never be stored directly

The platform needs product catalog, shopping cart, inventory,
checkout, payments, recommendations, and order management services.

Help me design the architecture while respecting these requirements.
"""

messages = [
    {
        "role": "user",
        "content": initial_prompt
    }
]

last_input_tokens = 0

### Build the Context Compaction Function

In [ ]:
# Low threshold for demonstration
COMPACTION_THRESHOLD = 5000

# Keep the latest 4 messages unchanged
RECENT_MESSAGES_TO_KEEP = 2


def compact_context(messages):

    conversation = "\n\n".join(
        f"{message['role'].upper()}:\n{message['content']}"
        for message in messages
    )

    prompt = f"""
            You are performing context compaction for a long-running AI conversation.

            Compress the conversation below while preserving everything needed
            to continue the task.

            PRESERVE:
            - Original user requirements
            - Important facts and numbers
            - Technical and architecture decisions
            - Constraints
            - Current task state
            - Unresolved issues

            REMOVE:
            - Repetition
            - Conversational filler
            - Verbose explanations
            - Redundant examples

            Do not invent information.

            Return a concise structured summary.

            CONVERSATION:

            {conversation}
    """

    response = client.messages.create(
        model=MODEL,
        max_tokens=1500,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return "".join(
        block.text
        for block in response.content
        if block.type == "text"
    )

### Create the Conversation Loop

In [ ]:
def chat(user_message):

    global messages
    global last_input_tokens

    # Check previous API token usage
    if last_input_tokens >= COMPACTION_THRESHOLD:

        print("\n **Context threshold reached — compacting...")

        old_messages = messages[:-RECENT_MESSAGES_TO_KEEP]
        recent_messages = messages[-RECENT_MESSAGES_TO_KEEP:]

        summary = compact_context(old_messages)

        messages = [
            {
                "role": "user",
                "content": f"""
                The following is compacted context from our earlier conversation.

                Treat it as authoritative project state.

                {summary}
                """
            }
        ] + recent_messages

        print("Context compaction complete.")

        # Reset because the conversation has just been compacted.
        # The next API response will give us the new actual input size.
        last_input_tokens = 0

    # Add new user message
    messages.append({
        "role": "user",
        "content": user_message
    })

    # Claude Claude Model with an API Call
    response = client.messages.create(
        model=MODEL,
        max_tokens=5000,
        messages=messages
    )

    answer = "".join(
        block.text
        for block in response.content
        if block.type == "text"
    )

    # Get actual token usage from the API response
    input_tokens = response.usage.input_tokens
    output_tokens = response.usage.output_tokens
    total_tokens = input_tokens + output_tokens

    last_input_tokens = total_tokens

    # Store response
    messages.append({
        "role": "assistant",
        "content": answer
    })

    print("\nASSISTANT:\n")
    print(answer)

    print("\n" + "-" * 60)
    print("TOKEN USAGE")
    print("-" * 60)

    print(f"Input tokens  : {input_tokens}")
    print(f"Output tokens : {output_tokens}")
    print(f"Total tokens  : {total_tokens}")

    return answer

### Simulate Conversations

In [ ]:
user_query = """Design the overall API and microservices architecture for Project Falcon.
Explain the responsibilities of each major service and how they communicate."""

chat(user_query)

In [ ]:
user_query = """Now design the product catalog service in detail.
Explain its APIs, database interactions, caching strategy,
scaling approach, and failure handling."""

chat(user_query)

In [ ]:
user_query = """Design the shopping cart service.
Explain how cart state should be managed, how Redis should be used,
how carts expire, and how the service should scale."""

chat(user_query)

In [ ]:
user_query = """Now design inventory management across multiple warehouses.
Discuss inventory reservations, consistency, concurrency,
failure scenarios, and communication with other services."""

chat(user_query)